# NTL Word-choice C0/C1/C2 — official-only, checkpointable CPU/Colab pilot
This is a **mechanism falsification**, not a champion score estimate. No external synthetic model outputs. No Kaggle submission. Free Colab interactive execution only; no unattended remote worker. It uses exactly the same frozen prompt folds, training rows, labels, HashingVectorizer dimension, SGD loss/alpha/epochs and random seeds across three arms. C0 unigrams, C1 unigrams plus adjacent context, C2 unigrams plus within-passage shuffled adjacent context. This is a *linear pilot*, **not** the previously proposed RoBERTa/LoRA reader. Do not claim model capacity equality in effective features; feature dimension and optimizer are held constant, while feature distributions differ.

In [ ]:
from pathlib import Path
import os, json, hashlib, random, collections, importlib.util
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import balanced_accuracy_score
import joblib
# Run interactively in Colab after cloning the repo (public source only):
# !git clone --branch research/attribution-bottleneck-review-20261106 https://github.com/kolihhan/name-kaggle.git
REPO=Path('/content/name-kaggle') if Path('/content/name-kaggle').exists() else Path('.')
DATA=next((p for p in (Path('/kaggle/input/competitions/name-that-llm/train.csv'),Path('/content/ntl_private/train.csv')) if p.is_file()),Path('/content/ntl_private/train.csv'))
FOLDS=REPO/'artifacts/phase1/folds.csv' # exact repository frozen row+prompt+model 5-fold CSV, not external guessed JSON
OUT=Path('/content/ntl_private/wordchoice_pilot')
OUT.mkdir(parents=True,exist_ok=True)
SEED=20261121; N_FEATURES=2**18; EPOCHS=3; ALPHA=1e-5
spec=importlib.util.spec_from_file_location('ntl_gate',REPO/'scripts/ntl_context_wordchoice_gate_20261121.py')
gate_mod=importlib.util.module_from_spec(spec); spec.loader.exec_module(gate_mod)
print('CPU/GPU:',os.environ.get('COLAB_GPU','CPU'), 'DATA present:',DATA.exists(), 'FOLDS present:',FOLDS.exists())


In [ ]:
def normalize_words(text):
    return str(text).split()

def context_features(text, arm, row_id):
    words=normalize_words(text)
    if not words: return 'EMPTY'
    # A deterministic token permutation destroys adjacency but keeps each document's unigram inventory.
    if arm=='C2':
        rng=random.Random(int.from_bytes(hashlib.sha256((str(SEED)+'|'+str(row_id)).encode()).digest()[:8],'big'))
        words=words.copy(); rng.shuffle(words)
    # C0 is strictly bag-of-words; C1 and C2 use the same capacity / feature templates.
    uni=['u:'+w for w in normalize_words(text)]
    if arm=='C0': return ' '.join(uni)
    return ' '.join(uni+['b:'+a+'~'+b for a,b in zip(words,words[1:])])

def slice_fingerprint(frame):
    """Hash ordered row IDs, prompt IDs, labels and raw text; no text is exported."""
    h=hashlib.sha256()
    for row in frame[['id','prompt_id','label','text']].itertuples(index=False, name=None):
        payload=json.dumps([str(row[0]),str(row[1]),int(row[2]),str(row[3])],
                           ensure_ascii=False,separators=(',',':')).encode('utf-8')
        h.update(len(payload).to_bytes(8,'big'))
        h.update(payload)
    return h.hexdigest()

def fit_one(train, valid, arm, fold):
    train_fp=slice_fingerprint(train)
    valid_fp=slice_fingerprint(valid)
    dest=OUT/f'{arm}_fold{fold}.joblib'
    # Resume only if the checkpoint matches the actual training/validation slices and experiment contract.
    if dest.exists():
        saved=joblib.load(dest)
        if (saved.get('contract')==[SEED,N_FEATURES,EPOCHS,ALPHA] and
            saved.get('train_ids')==train.id.astype(str).tolist() and
            saved.get('heldout_ids')==valid.id.astype(str).tolist() and
            saved.get('train_fingerprint')==train_fp and
            saved.get('heldout_fingerprint')==valid_fp and
            saved.get('arm')==arm and saved.get('fold')==int(fold)):
            return saved
        raise ValueError(f'Stale checkpoint {dest}; move it aside manually before retrying')
    vectorizer=HashingVectorizer(n_features=N_FEATURES,alternate_sign=False,norm='l2',tokenizer=str.split,token_pattern=None,lowercase=False)
    # Fixed class set and identical optimization. No use of held-out labels during fitting.
    xtrain=vectorizer.transform(context_features(t,arm,i) for t,i in zip(train.text,train.id))
    xvalid=vectorizer.transform(context_features(t,arm,i) for t,i in zip(valid.text,valid.id))
    model=SGDClassifier(loss='log_loss',alpha=ALPHA,max_iter=EPOCHS,tol=None,random_state=SEED,shuffle=True)
    model.fit(xtrain,train.label.astype(int))
    pred=model.predict(xvalid).astype(int)
    checkpoint={'model':model,'fold':int(fold),'arm':arm,'train_ids':train.id.astype(str).tolist(),
                'train_prompt_ids':sorted(set(train.prompt_id.astype(str))),
                'heldout_ids':valid.id.astype(str).tolist(),'pred':pred.tolist(),
                'heldout_row_ids':valid.id.astype(str).tolist(), 'contract':[SEED,N_FEATURES,EPOCHS,ALPHA],
                'train_fingerprint':train_fp,'heldout_fingerprint':valid_fp}
    joblib.dump(checkpoint,dest)
    return checkpoint

def pilot(frame, folds, family):
    # Build manifest from the ACTUAL fit slices, never from a post-hoc complement constructor.
    frame=frame.copy()
    frame['id']=frame.id.astype(str); frame['prompt_id']=frame.prompt_id.astype(str)
    frame['fold']=frame.prompt_id.map(folds)
    assert frame.fold.notna().all() and frame.id.is_unique
    assert set(frame.label.astype(int))==set(range(18))
    manifests={a:{} for a in ('C0','C1','C2')}
    preds={a:{} for a in manifests}
    for fold in sorted(set(folds.values())):
        train=frame[frame.fold!=fold].copy()
        valid=frame[frame.fold==fold].copy()
        assert not set(train.prompt_id)&set(valid.prompt_id)
        assert set(train.label.astype(int))==set(range(18))
        for arm in manifests:
            ck=fit_one(train,valid,arm,fold)
            manifests[arm][int(fold)]={k:ck[k] for k in ('train_ids','train_prompt_ids','heldout_ids')}
            preds[arm].update(zip(ck['heldout_row_ids'],ck['pred']))
    records=[]
    for row in frame.itertuples(index=False):
        records.append(dict(id=row.id,prompt_id=row.prompt_id,fold=int(row.fold),
                            source='official-dev',label=int(row.label),
                            **{a:int(preds[a][row.id]) for a in preds}))
    gate_mod.validate_fit_provenance(records,folds,manifests)
    # The existing gate's sibling_macro is sibling-CLASS recall, not conditional pair discrimination.
    # A separate true pair-conditioned diagnostic requires knowing the top-two class scores.
    # Caller supplies real model-family map for official data, synthetic-only map for dry-run.
    result=gate_mod.gate(records,folds,family,manifests)
    # No placeholder family grouping is permitted for a real experiment.
    (OUT/'manifest.json').write_text(json.dumps(manifests,indent=2))
    (OUT/'predictions.json').write_text(json.dumps(records,indent=2))
    (OUT/'metrics.json').write_text(json.dumps(result,indent=2))
    return result


In [ ]:
# CPU dry-run: exercises real fit_one train indices, not post-hoc manifests.
# Uses 5 folds x 18 classes x 2 examples, with disjoint synthetic prompts.
def synthetic():
    rows=[]; folds={}
    for f in range(5):
        for p in range(2):
            prompt=f'synth_{f}_{p}'; folds[prompt]=f
            for y in range(18):
                rows.append(dict(id=f'{prompt}_{y}',prompt_id=prompt,label=y,
                    text=f'common topic_{p} author_{y} token_{y} token_{y} sentence end'))
    return pd.DataFrame(rows),folds
synth,synth_folds=synthetic()
res=pilot(synth,synth_folds,{i:i//2 for i in range(18)}) # synthetic-only arbitrary families
assert len(json.loads((OUT/'predictions.json').read_text()))==180
assert len(json.loads((OUT/'manifest.json').read_text())['C1'])==5
print('DRY_RUN_OK: fit-derived manifests, 3 arms x 5 folds, 180 heldout rows; synthetic metrics NOT performance evidence')

# Adversarial resume regression: same row IDs, changed training text or labels must FAIL.
# This exercises the exact committed fit_one, not a post-hoc manifest.
_probe_fold=0
_probe_train=synth[synth.prompt_id.map(synth_folds)!=_probe_fold].copy()
_probe_valid=synth[synth.prompt_id.map(synth_folds)==_probe_fold].copy()
original=fit_one(_probe_train,_probe_valid,'C1',_probe_fold)
assert original['train_fingerprint']==slice_fingerprint(_probe_train)
assert original['heldout_fingerprint']==slice_fingerprint(_probe_valid)
for field, replacement in [('text','MUTATED_WITH_IDENTICAL_ID'),('label',(_probe_train.iloc[0].label+1)%18)]:
    mutated=_probe_train.copy()
    mutated.loc[mutated.index[0],field]=replacement
    try:
        fit_one(mutated,_probe_valid,'C1',_probe_fold)
        raise AssertionError('STALE_CHECKPOINT_ACCEPTED:'+field)
    except ValueError as exc:
        assert 'Stale checkpoint' in str(exc), str(exc)
changed_valid=_probe_valid.copy()
changed_valid.loc[changed_valid.index[0],'text']='MUTATED_HELDOUT_WITH_IDENTICAL_ID'
try:
    fit_one(_probe_train,changed_valid,'C1',_probe_fold)
    raise AssertionError('STALE_HELDOUT_ACCEPTED')
except ValueError as exc:
    assert 'Stale checkpoint' in str(exc),str(exc)
print('CHECKPOINT_MUTATION_REJECT_OK: train text, train label, heldout text')


In [ ]:
# REAL DATA: load official Kaggle input only; use frozen row+prompt+model folds.csv.
# This experiment is exploratory on spent official folds and NOT strong-champion validation.
if DATA.is_file() and FOLDS.is_file():
    import sys
    sys.path.insert(0,str(REPO))
    from scripts.ntl_official_pilot_input_20261121 import load_official
    official,frozen,real_families,model_labels=load_official(DATA,FOLDS)
    assert len(model_labels)==18 and len(set(real_families.values()))==6
    OUT=Path('/kaggle/working/wordchoice_official') if Path('/kaggle/working').exists() else Path('/content/ntl_private/wordchoice_official')
    OUT.mkdir(parents=True,exist_ok=True)
    print('OFFICIAL_FOLD_READY:',len(official),'classes',len(model_labels),'prompts',len(frozen))
    result=pilot(official,frozen,real_families)
    print('EXPLORE_ONLY: frozen official 5-fold same-source BA; not evidence vs 0.702 Champion')
    print(result)
else:
    print('OFFICIAL_DATA_NOT_MOUNTED: add the Kaggle competition data within Kaggle or supply it in private Colab; NO real training occurred.')
